# CatBoost 建模主流程（防泄漏 & 泛化增强）

**要点**：
- 仅使用与 `record_time` 对齐的相对时间差特征；删除原始时间戳。
- 类别列直接作为类别传入 CatBoost。
- 5 折 StratifiedKFold + 早停 + OOF AUC 评估；最后用折内最优迭代数均值复训全量再推理测试集。
- 自动设置 `scale_pos_weight` 应对类不平衡。

In [12]:
# 如需在新环境运行，可先安装依赖（本环境若已安装，可跳过）
# !pip install catboost scikit-learn pandas numpy -q

In [13]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier, Pool
SEED = 2025
np.random.seed(SEED)
NON_FEATURE_COLS = ['id','label']

In [14]:
# ===== 路径配置（请按实际环境修改） =====
TRAIN_CSV = 'train/train.csv'
TRAIN_STMT_FEAT = 'train/train_statement_feature_v2.csv'
TEST_CSV = 'testaa/testaa.csv'                      # 如果没有可置为 None
TEST_STMT_FEAT = 'testaa/testaa_statement_feature_v2.csv'  # 如果没有可置为 None
OUT_SUBMISSION = 'output_Cat/submission.csv'
SAVE_OOF = 'output_Cat/oof_pred.csv'
SAVE_INFO = 'output_Cat/cv_info.json'

In [15]:
def _coerce_numeric(series):
    return pd.to_numeric(series, errors='coerce')

def _safe_div(a, b):
    b = b.copy() if hasattr(b,'copy') else b
    try:
        b = b.replace(0, np.nan)
    except Exception:
        b = np.where(b==0, np.nan, b)
    return a / b

def base_feature_engineering(df: pd.DataFrame, is_train=True):
    X = df.copy()
    for col in X.columns:
        if X[col].dtype == 'O':
            X[col] = X[col].replace(['', ' ', 'nan', 'NaN', 'NULL', 'None'], np.nan)
    for c in ['title','career','zip_code','residence','term','syndicated','installment','level']:
        if c in X.columns:
            if c=='level':
                X[c] = X[c].astype('object')
            else:
                X[c] = pd.to_numeric(X[c], errors='coerce').astype('Int64')
    if set(['record_time','issue_time']).issubset(X.columns):
        X['days_issue_to_record'] = ((X['record_time']-X['issue_time'])/86400.0).clip(lower=0)
    if set(['record_time','history_time']).issubset(X.columns):
        X['days_history_to_record'] = ((X['record_time']-X['history_time'])/86400.0).clip(lower=0)
    if set(['balance','balance_limit']).issubset(X.columns):
        X['balance_utilization'] = _safe_div(_coerce_numeric(X['balance']), _coerce_numeric(X['balance_limit']))
    if set(['balance_accounts','total_accounts']).issubset(X.columns):
        X['acct_utilization'] = _safe_div(_coerce_numeric(X['balance_accounts']), _coerce_numeric(X['total_accounts']))
    for col in ['issue_time','record_time','history_time']:
        if col in X.columns:
            X.drop(columns=[col], inplace=True)
    return X

def prepare_dataset(train_csv, stmt_feat_csv, test_csv=None, test_stmt_feat_csv=None):
    train = pd.read_csv(train_csv)
    stmt = pd.read_csv(stmt_feat_csv)
    df = train.merge(stmt, on='id', how='left', suffixes=('', '_stmt'))
    df['stmt_missing'] = df['has_statement'].apply(lambda x: 0 if x==1 else 1) if 'has_statement' in df.columns else 1
    y = df['label'].astype(int)
    X = df.drop(columns=['label'])
    X_test = None
    if test_csv is not None and os.path.exists(test_csv):
        test = pd.read_csv(test_csv)
        if test_stmt_feat_csv is not None and os.path.exists(test_stmt_feat_csv):
            stmt_t = pd.read_csv(test_stmt_feat_csv)
            X_test = test.merge(stmt_t, on='id', how='left', suffixes=('', '_stmt'))
            X_test['stmt_missing'] = X_test['has_statement'].apply(lambda x: 0 if x==1 else 1) if 'has_statement' in X_test.columns else 1
        else:
            X_test = test.copy()
            X_test['stmt_missing'] = 1
    return X, y, X_test

def get_cat_cols(df: pd.DataFrame):
    cat_cols = []
    for c in ['title','career','zip_code','residence','term','syndicated','installment','level']:
        if c in df.columns:
            cat_cols.append(c)
    for c in df.select_dtypes(include='object').columns:
        if c not in cat_cols:
            cat_cols.append(c)
    cat_cols = [c for c in cat_cols if c not in NON_FEATURE_COLS and c in df.columns]
    return sorted(list(set(cat_cols)))

def fill_missing(df: pd.DataFrame, cat_cols):
    X = df.copy()
    for c in X.columns:
        if c in NON_FEATURE_COLS:
            continue
        if c in cat_cols:
            X[c] = X[c].astype('object').fillna('Unknown')
        else:
            if X[c].dtype == 'O':
                xnum = pd.to_numeric(X[c], errors='coerce')
                if xnum.notna().sum()>0:
                    X[c] = xnum
            X[c] = X[c].fillna(X[c].median())
    return X

def run_cv_catboost(X, y, X_test=None, params=None, n_splits=5, seed=SEED, early_stopping=200):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    cat_cols = get_cat_cols(X)
    X = fill_missing(base_feature_engineering(X), cat_cols)
    if X_test is not None:
        X_test = fill_missing(base_feature_engineering(X_test, is_train=False), cat_cols)
    def to_pool(df, y=None):
        cat_idx = [df.columns.get_loc(c) for c in cat_cols if c in df.columns]
        return Pool(df, label=y, cat_features=cat_idx)
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test)) if X_test is not None else None
    best_iterations, fold_auc = [], []
    default_params = dict(
        loss_function='Logloss', eval_metric='AUC', iterations=5000, learning_rate=0.03,
        depth=6, l2_leaf_reg=6.0, random_seed=seed, bootstrap_type='Bayesian',
        bagging_temperature=0.25, border_count=128, grow_policy='SymmetricTree',
        od_type='Iter', od_wait=early_stopping, verbose=200, allow_writing_files=False
    )
    pos, neg = int((y==1).sum()), int((y==0).sum())
    default_params['scale_pos_weight'] = max(1.0, neg/max(1,pos))
    if params: default_params.update(params)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        model = CatBoostClassifier(**default_params)
        model.fit(to_pool(X_tr, y_tr), eval_set=to_pool(X_va, y_va), verbose=200)
        va_pred = model.predict_proba(X_va)[:,1]
        oof[va_idx] = va_pred
        va_auc = roc_auc_score(y_va, va_pred)
        fold_auc.append(va_auc)
        best_iterations.append(model.get_best_iteration())
        if X_test is not None:
            test_pred += model.predict_proba(X_test)[:,1] / n_splits
        print(f"[Fold {fold}] AUC = {va_auc:.6f} | best_iter = {model.get_best_iteration()}")
    cv_auc = roc_auc_score(y, oof)
    print(f"OOF AUC = {cv_auc:.6f}; folds = {[round(a,6) for a in fold_auc]}")
    return dict(oof=oof, test_pred=test_pred, cv_auc=cv_auc, fold_auc=fold_auc,
                best_iterations=best_iterations, cat_cols=cat_cols)

def retrain_full_and_predict(X, y, X_test, cat_cols, best_iterations, params=None, seed=SEED):
    if X_test is None:
        return None
    best_iter = int(np.mean(best_iterations))
    default_params = dict(
        loss_function='Logloss', eval_metric='AUC', iterations=max(100,best_iter), learning_rate=0.03,
        depth=6, l2_leaf_reg=6.0, random_seed=seed, bootstrap_type='Bayesian',
        bagging_temperature=0.25, border_count=128, grow_policy='SymmetricTree', verbose=False,
        allow_writing_files=False
    )
    if params: default_params.update(params)
    X_proc = fill_missing(base_feature_engineering(X), cat_cols)
    X_test_proc = fill_missing(base_feature_engineering(X_test, is_train=False), cat_cols)
    cat_idx = [X_proc.columns.get_loc(c) for c in cat_cols if c in X_proc.columns]
    model = CatBoostClassifier(**default_params)
    model.fit(Pool(X_proc, label=y, cat_features=cat_idx), verbose=200)
    preds = model.predict_proba(X_test_proc)[:,1]
    return preds

In [17]:
X, y, X_test = prepare_dataset(TRAIN_CSV, TRAIN_STMT_FEAT, TEST_CSV, TEST_STMT_FEAT)

In [18]:
X

,id,title,career,zip_code,residence,loan,term,interest_rate,issue_time,syndicated,...,amt_p90,net_flow,income_expense_ratio,tx_per_active_day,record_time_stmt,last_tx_days_before_record,first_tx_days_before_record,span_days,has_statement,stmt_missing
0,0,9,0.0,221373,1,7200,36,10.95,1238631967,0,...,9082.000,47628.00,4.942878,1.170732,1238630622,5.002569,168.002569,163.0,1,0
1,1,8,10.0,311681,0,21300,36,12.95,1128212052,0,...,1904.776,6543.32,1.311450,1.130952,1161907665,12.037280,180.033009,174.0,0,1
2,2,8,7.0,271562,1,10400,60,21.05,1249171509,0,...,1545.063,-9361.34,0.410633,1.043478,1383958593,0.039271,180.039271,180.0,1,0
3,3,7,2.0,522083,0,33050,36,16.40,1172882234,0,...,1904.776,6543.32,1.311450,1.130952,1214353935,12.037280,180.033009,174.0,0,1
4,4,8,3.0,101026,1,5200,36,14.35,1172882384,0,...,1717.074,-10483.20,0.659891,1.120482,1240274527,23.029248,192.029248,169.0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53475,53475,2,2.0,603000,1,9000,12,23.55,1172880000,0,...,1904.776,6543.32,1.311450,1.130952,1157587200,12.037280,180.033009,174.0,0,1
53476,53476,0,10.0,601702,1,8000,12,30.70,1160092800,0,...,1904.776,6543.32,1.311450,1.130952,1138665600,12.037280,180.033009,174.0,0,1
53477,53477,2,10.0,602808,1,10000,12,9.40,1180310400,0,...,1904.776,6543.32,1.311450,1.130952,1108771200,12.037280,180.033009,174.0,0,1
53478,53478,0,10.0,602102,2,9000,12,24.40,1176768000,0,...,1904.776,6543.32,1.311450,1.130952,1159660800,12.037280,180.033009,174.0,0,1


In [16]:
# === 运行：准备数据 & 交叉验证 & 生成提交 ===

X, y, X_test = prepare_dataset(TRAIN_CSV, TRAIN_STMT_FEAT, TEST_CSV, TEST_STMT_FEAT)
cv = run_cv_catboost(X, y, X_test=X_test)
oof_df = pd.DataFrame({'id': X['id'].values, 'oof_pred': cv['oof'], 'label': y.values})
oof_df.to_csv(SAVE_OOF, index=False, encoding='utf-8')
print('OOF 保存：', SAVE_OOF, oof_df.shape)
if X_test is not None:
    preds = retrain_full_and_predict(X, y, X_test, cv['cat_cols'], cv['best_iterations'])
    sub = pd.DataFrame({'id': X_test['id'].values, 'prob': preds})
    sub.to_csv(OUT_SUBMISSION, index=False, encoding='utf-8')
    print('提交文件保存：', OUT_SUBMISSION, sub.shape)
info = {'cv_auc': float(cv['cv_auc']), 'fold_auc': [float(x) for x in cv['fold_auc']],
        'best_iterations': [int(x) for x in cv['best_iterations']], 'cat_cols': cv['cat_cols'], 'seed': SEED}
with open(SAVE_INFO, 'w', encoding='utf-8') as f:
    json.dump(info, f, ensure_ascii=False, indent=2)
print('CV 信息保存：', SAVE_INFO)

0:	test: 0.6313259	best: 0.6313259 (0)	total: 338ms	remaining: 28m 9s
200:	test: 0.6571086	best: 0.6572201 (192)	total: 10.9s	remaining: 4m 19s
400:	test: 0.6567425	best: 0.6572376 (254)	total: 19.1s	remaining: 3m 39s
600:	test: 0.6566165	best: 0.6574390 (449)	total: 28.4s	remaining: 3m 27s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.6574390326
bestIteration = 449

Shrink model to first 450 iterations.
[Fold 1] AUC = 0.657439 | best_iter = 449
0:	test: 0.6184841	best: 0.6184841 (0)	total: 61.4ms	remaining: 5m 6s
200:	test: 0.6570109	best: 0.6570109 (200)	total: 10.5s	remaining: 4m 9s
400:	test: 0.6576494	best: 0.6576494 (400)	total: 20.3s	remaining: 3m 52s
600:	test: 0.6591127	best: 0.6594475 (537)	total: 31.8s	remaining: 3m 52s
800:	test: 0.6587109	best: 0.6595612 (640)	total: 41.3s	remaining: 3m 36s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.6595612127
bestIteration = 640

Shrink model to first 641 iterations.
[Fold 2] AUC = 0.659561

**调参建议**：如 OOF 曲线震荡明显或过拟合，可提升 `l2_leaf_reg`、降低 `learning_rate`，或将 `grow_policy='Lossguide'`，同时适度调高 `bagging_temperature`。